In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# Sigma — output scales, $\sigma_i^2$, $L_i^{floor}$ and the reference floor $r$

Algorithm A1 (iv)-(v) of r05 over every family-stage and seed: $D_\phi$, $Y^{min}$,
$Y^{max}$ (Eq. 3) and $\sigma_i^2$, $L_i^{floor}$ (Eq. 23) from each seed's training
devices only, plus $r = Q_{0.95}$ of the refinement gap (Eq. 21). Per-device sums are
computed once and averaged per seed. Finished family-stages are skipped, so re-running
this notebook after the `sigma` step only reads.

Writes `PINN-DPV/Results/sigma/<family-stage>_seed<k>.json`.

In [ ]:
import sys

sys.path.insert(0, str(ROOT / "src"))
# The fingerprint of the code this notebook ran against; `verify` recomputes it.
from PINN_DPV import report_digest

print(report_digest.stamp())


In [ ]:
import os

from PINN_DPV import data, steps

# $PINN_DPV_FAMILY_STAGES (e.g. "E1A,E3A") limits this run to those family-stages.
FAMILY_STAGES = tuple(f for f in os.environ.get("PINN_DPV_FAMILY_STAGES", "").split(",") if f) or steps.FAMILY_STAGES
missing = [fs for fs in FAMILY_STAGES if not all(Path(p).exists() for p in data.canonical_paths(fs))]
assert not missing, f"dPV files not found under {data.dataset_root()} for {missing}"
print("results:", steps.SIGMA_DIR)
paths = steps.run_sigma(FAMILY_STAGES)
print(len(paths), "sigma files")

## What the scaling says

`floor/σ²` near 1 means dPV's own discretization already accounts for most of the
residual scale: that component cannot discriminate errors below it. `r` is the
reference floor every e_PCE is read against.

In [ ]:
import json

import numpy as np

rows = {}
for fs in FAMILY_STAGES:
    recs = [json.loads(Path(steps.SIGMA_DIR, f"{fs}_seed{s}.json").read_text()) for s in steps.SEEDS]
    comps = sorted(recs[0]["sigma2"])
    ratio = {c: float(np.median([r["floor"][c] / r["sigma2"][c] for r in recs])) for c in comps}
    rows[fs] = (recs[0]["reference_floor"]["r_pp"], recs[0]["n_train"], recs[0]["n_devices"], ratio)

comps = sorted({c for v in rows.values() for c in v[3]})
print(f"{'':6}{'r (pp)':>8}{'train':>7}{'dev':>6}  " + "".join(f"{c:>12}" for c in comps))
for fs, (r, ntr, nd, ratio) in rows.items():
    print(f"{fs:6}{r:8.4f}{ntr:7d}{nd:6d}  " + "".join(f"{ratio.get(c, float('nan')):12.2e}" for c in comps))